# I tensori: i numeri del modello, e la forma che hanno

*Anatomia di un LLM* · capitolo 3, I mattoni · script `01-01`, `scripts/01_01_tensori_e_shape.py`

Tutto quello che entra, esce e si impara in un modello e' una griglia di numeri
con una forma: si chiama tensore. Un numero solo, una fila, una tabella, una
pila di tabelle. Quasi tutti gli errori che si fanno costruendo un modello
sono errori di forma, e quasi tutta la fatica di leggerne uno e' seguire come
la forma cambia: vale la pena spenderci una pagina prima di cominciare.

Si esegue con `python3 scripts/01_01_tensori_e_shape.py`
e dura circa 5 secondi.

> **Per riottenere i numeri del libro** esegui le celle in ordine, dalla prima. Il seme si fissa una volta sola, in `prepara()`: rieseguire una cella a metà cambia i numeri casuali di tutte quelle dopo.

[Apri in Colab](https://colab.research.google.com/github/fcollova/llm-anatomy/blob/main/notebooks/01_01_tensori_e_shape.ipynb) · la prima cella scarica il codice, se non lo trova accanto.

In [1]:
# Porta il notebook nella cartella che contiene llm_anatomy/, da dovunque
# sia stato aperto: gli script si eseguono da li', e cosi' anche le celle.
import os
import subprocess
import sys
from pathlib import Path

radice = next((p for p in [Path.cwd(), *Path.cwd().parents]
               if (p / "llm_anatomy").is_dir()), None)

if radice is None:
    # Il notebook e' stato aperto da solo, per esempio su Colab: il codice, il
    # testo e il modello addestrato si scaricano una volta, qui accanto.
    REPO, RAMO, CARTELLA = "fcollova/llm-anatomy", "main", ""
    copia = Path("anatomia-di-un-llm")
    if not copia.exists():
        esito = subprocess.run(
            ["git", "clone", "-q", "--depth", "1", "--branch", RAMO,
             f"https://github.com/{REPO}.git", str(copia)], capture_output=True)
        if esito.returncode != 0:
            raise SystemExit("Non riesco a scaricare il codice da github.com/"
                             + REPO + ": " + esito.stderr.decode()[-300:])
    radice = copia / CARTELLA

os.chdir(radice)
sys.path.insert(0, str(Path.cwd()))

In [2]:
import torch

from llm_anatomy.utils import prepara, stampa_shape, titolo, verifica

prepara()

[ambiente] torch 2.13.0+cpu  device=cpu  threads=8  seed=1337


device(type='cpu')

In [3]:
titolo("Un numero, una fila, una tabella")
numero = torch.tensor(3.0)
fila = torch.tensor([1.0, 2.0, 3.0])
tabella = torch.tensor([[1.0, 2.0, 3.0],
                        [4.0, 5.0, 6.0]])
for nome, t in (("numero", numero), ("fila", fila), ("tabella", tabella)):
    print(f"  {nome:<8} assi {t.ndim}   forma {tuple(t.shape)}")
print()
print(tabella)


Un numero, una fila, una tabella
--------------------------------
  numero   assi 0   forma ()
  fila     assi 1   forma (3,)
  tabella  assi 2   forma (2, 3)

tensor([[1., 2., 3.],
        [4., 5., 6.]])


In [4]:
titolo("Una pila di tabelle")
B, T, C = 2, 5, 4    # frasi, caratteri, numeri per carattere
x = torch.arange(B * T * C, dtype=torch.float32).reshape(B, T, C)
stampa_shape("x (B, T, C)", x)
print("\n  la prima frase, x[0]: cinque righe, una per carattere\n")
print(x[0])


Una pila di tabelle
-------------------
  x (B, T, C)                  (2, 5, 4)              float32

  la prima frase, x[0]: cinque righe, una per carattere

tensor([[ 0.,  1.,  2.,  3.],
        [ 4.,  5.,  6.,  7.],
        [ 8.,  9., 10., 11.],
        [12., 13., 14., 15.],
        [16., 17., 18., 19.]])


In [5]:
titolo("Prendere un pezzo")
stampa_shape("x[0]      una frase", x[0])
stampa_shape("x[0, 2]   un carattere", x[0, 2])
stampa_shape("x[:, -1]  ultimo carattere", x[:, -1])
print(f"  x[0, 2] = {x[0, 2].tolist()}")


Prendere un pezzo
-----------------
  x[0]      una frase          (5, 4)                 float32
  x[0, 2]   un carattere       (4,)                   float32
  x[:, -1]  ultimo carattere   (2, 4)                 float32
  x[0, 2] = [8.0, 9.0, 10.0, 11.0]


In [6]:
titolo("Cambiare forma")
piatto = x.view(B * T, C)
stampa_shape("x.view(B*T, C)", piatto)
verifica(torch.equal(piatto[7], x[1, 2]),
         "la riga 7 della tabella piatta e' il carattere 2 della frase 1")
n_head, d_head = 2, C // 2
teste = x.view(B, T, n_head, d_head).transpose(1, 2)
stampa_shape("(B, T, C) -> (B, h, T, d_h)", teste)
ricomposto = teste.transpose(1, 2).contiguous().view(B, T, C)
verifica(torch.equal(ricomposto, x),
         "divisa in teste e ricomposta, x torna identica")


Cambiare forma
--------------
  x.view(B*T, C)               (10, 4)                float32
  [verifica superata] la riga 7 della tabella piatta e' il carattere 2 della
                      frase 1
  (B, T, C) -> (B, h, T, d_h)  (2, 2, 5, 2)           float32
  [verifica superata] divisa in teste e ricomposta, x torna identica


In [7]:
titolo("Sommare una cosa piccola a una grande")
pos = torch.tensor([100.0, 200.0, 300.0, 400.0])
somma = x + pos
stampa_shape("pos (C,)", pos)
stampa_shape("x + pos", somma)
print(f"  somma[0, 0] = {somma[0, 0].tolist()}")
print(f"  somma[1, 4] = {somma[1, 4].tolist()}")
print("  regola: le forme si allineano da destra,")
print("  (2, 5, 4) e (4,) -> (2, 5, 4)")
verifica(torch.equal(somma[1, 4], x[1, 4] + pos),
         "la stessa fila pos e' stata sommata a ogni carattere di ogni frase")


Sommare una cosa piccola a una grande
-------------------------------------
  pos (C,)                     (4,)                   float32
  x + pos                      (2, 5, 4)              float32
  somma[0, 0] = [100.0, 201.0, 302.0, 403.0]
  somma[1, 4] = [136.0, 237.0, 338.0, 439.0]
  regola: le forme si allineano da destra,
  (2, 5, 4) e (4,) -> (2, 5, 4)
  [verifica superata] la stessa fila pos e' stata sommata a ogni carattere di
                      ogni frase


## Da provare

cambia B, T e C nella seconda sezione e guarda come cambiano le forme.